# Shabaka Pulse — Priority Dispatch Solver

This notebook implements and validates the priority-based demand-response dispatch algorithm for allocating renewable energy surplus across industrial facilities.

### Priority Order Rationale:
1. **Tier 1 (Hydraulic Storage)**: Desalination plants and water pumping stations. High ramp rates and fast response times; water storage reservoirs buffer operations so load shedding causes zero risk of product or inventory damage.
2. **Tier 2 (Thermal Storage)**: Cold storage logistics and food preservation facilities. Moderate ramp rates; thermal inertia in refrigeration systems provides a buffer window before food safety bounds are breached.
3. **Tier 3 (Batch Processing)**: Cement grinding mills and steel rolling mills. Slower ramp rates and high startup overhead; held in reserve to avoid interrupting active material batch cycles. Surplus is allocated to Tier 3 only after Tier 1 and Tier 2 capacities are fully saturated.

## 1. Setup & Load clustered dataset

Importing data processing and typing utilities, then loading `data/facilities_clustered.csv` to inspect facility tier assignments.

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple
import pandas as pd

# Load clustered facility dataset containing assigned operational tiers
data_path = Path("data/facilities_clustered.csv")
df_facilities = pd.read_csv(data_path)

display(df_facilities.head())
print(f"Loaded dataset shape: {df_facilities.shape[0]} rows, {df_facilities.shape[1]} columns")
display(df_facilities["tier"].value_counts().sort_index().to_frame())

,facility_id,facility_name,industry_type,max_flex_mw,ramp_rate_mw_per_min,storage_type,lat,lon,raw_cluster,tier
0,FAC-001,Desalination Plant #1,Desalination Plant,20.97,3.65,hydraulic,30.9270,29.4824,1,1
1,FAC-002,Food Preservation Facility #1,Food Preservation Facility,16.42,1.29,thermal,30.8792,29.5373,2,2
2,FAC-003,Steel Rolling Mill #1,Steel Rolling Mill,95.61,0.36,batch,29.9064,32.5734,0,3
3,FAC-004,Desalination Plant #2,Desalination Plant,23.86,1.66,hydraulic,24.4739,32.7375,2,2
4,FAC-005,Cold Storage Logistics #1,Cold Storage Logistics,10.32,1.47,thermal,30.8723,29.5586,2,2


Loaded dataset shape: 30 rows, 10 columns


,count
tier,
1,8
2,12
3,10


## 2. Implement the dispatch function

The `allocate_surplus` function receives an available renewable surplus amount (MW) and distributes it across facilities according to operational tier priority:
- Tier 1 facilities receive allocation first.
- Tier 2 facilities receive allocation second if surplus remains.
- Tier 3 facilities receive allocation last if surplus remains.

Within each tier, facilities are processed in deterministic order by `facility_id` to guarantee reproducible allocation results across simulation runs.

In [ ]:
def allocate_surplus(
    surplus_mw: float,
    clustered_facilities: pd.DataFrame,
) -> Tuple[Dict[str, float], float]:
    remaining_surplus = float(surplus_mw)
    allocations: Dict[str, float] = {
        str(fac_id): 0.0 for fac_id in clustered_facilities["facility_id"]
    }

    # Loop through tiers in priority order [1, 2, 3]
    for tier_num in [1, 2, 3]:
        if remaining_surplus <= 0.0:
            break

        # Filter facilities in the current tier and sort by facility_id for deterministic execution
        tier_facilities = clustered_facilities[
            clustered_facilities["tier"] == tier_num
        ].sort_values(by="facility_id")

        for _, row in tier_facilities.iterrows():
            if remaining_surplus <= 0.0:
                break

            fac_id = str(row["facility_id"])
            cap_mw = float(row["max_flex_mw"])

            # Allocate up to the facility's max flexible capacity
            allocated = min(cap_mw, remaining_surplus)
            allocations[fac_id] = round(allocated, 4)
            remaining_surplus -= allocated

    return allocations, round(remaining_surplus, 6)

## 3. Test at low surplus (100 MW)

Evaluating dispatch allocation under a low surplus scenario (100 MW). Expectation: The available surplus should be absorbed by Tier 1 facilities, leaving Tier 2 and Tier 3 untouched.

In [3]:
surplus_low = 100.0
allocations_low, remaining_low = allocate_surplus(surplus_low, df_facilities)

# Diagnostics
allocated_df_low = pd.DataFrame(
    [
        {
            "facility_id": fac_id,
            "allocated_mw": mw,
            "tier": df_facilities.loc[df_facilities["facility_id"] == fac_id, "tier"].values[0],
        }
        for fac_id, mw in allocations_low.items()
        if mw > 0.0
    ]
)

total_allocated_low = sum(allocations_low.values())

print(f"Input Surplus: {surplus_low} MW")
print(f"Facilities receiving allocation: {len(allocated_df_low)}")
print(f"Total Allocated MW: {total_allocated_low:.2f} MW")
print(f"Remaining Unallocated MW: {remaining_low:.2f} MW")

display(allocated_df_low["tier"].value_counts().to_frame())

# Conservation of energy assertion
assert abs((total_allocated_low + remaining_low) - surplus_low) < 1e-6, (
    f"Energy balance mismatch: {total_allocated_low} + {remaining_low} != {surplus_low}"
)

Input Surplus: 100.0 MW
Facilities receiving allocation: 5
Total Allocated MW: 100.00 MW
Remaining Unallocated MW: 0.00 MW


,count
tier,
1,5


## 4. Test at medium surplus (350 MW)

Evaluating dispatch allocation under a medium surplus scenario (350 MW). Expectation: Tier 1 capacity will be exhausted, and remaining power will spill over into Tier 2 facilities.

In [4]:
surplus_med = 350.0
allocations_med, remaining_med = allocate_surplus(surplus_med, df_facilities)

# Diagnostics
allocated_df_med = pd.DataFrame(
    [
        {
            "facility_id": fac_id,
            "allocated_mw": mw,
            "tier": df_facilities.loc[df_facilities["facility_id"] == fac_id, "tier"].values[0],
        }
        for fac_id, mw in allocations_med.items()
        if mw > 0.0
    ]
)

total_allocated_med = sum(allocations_med.values())

print(f"Input Surplus: {surplus_med} MW")
print(f"Facilities receiving allocation: {len(allocated_df_med)}")
print(f"Total Allocated MW: {total_allocated_med:.2f} MW")
print(f"Remaining Unallocated MW: {remaining_med:.2f} MW")

display(allocated_df_med["tier"].value_counts().to_frame())

# Conservation of energy assertion
assert abs((total_allocated_med + remaining_med) - surplus_med) < 1e-6, (
    f"Energy balance mismatch: {total_allocated_med} + {remaining_med} != {surplus_med}"
)

Input Surplus: 350.0 MW
Facilities receiving allocation: 20
Total Allocated MW: 350.00 MW
Remaining Unallocated MW: 0.00 MW


,count
tier,
2,12
1,8


## 5. Test at high surplus (900 MW)

Evaluating dispatch allocation under a high surplus scenario (900 MW). Expectation: Available surplus may exceed total system-wide flexible capacity across all 30 facilities, verifying that excess power remains unallocated without causing allocation overflow.

In [5]:
surplus_high = 900.0
allocations_high, remaining_high = allocate_surplus(surplus_high, df_facilities)

# Diagnostics
allocated_df_high = pd.DataFrame(
    [
        {
            "facility_id": fac_id,
            "allocated_mw": mw,
            "tier": df_facilities.loc[df_facilities["facility_id"] == fac_id, "tier"].values[0],
        }
        for fac_id, mw in allocations_high.items()
        if mw > 0.0
    ]
)

total_allocated_high = sum(allocations_high.values())
total_system_capacity = df_facilities["max_flex_mw"].sum()

print(f"Input Surplus: {surplus_high} MW")
print(f"Total System-Wide Flexible Capacity: {total_system_capacity:.2f} MW")
print(f"Facilities receiving allocation: {len(allocated_df_high)}")
print(f"Total Allocated MW: {total_allocated_high:.2f} MW")
print(f"Remaining Unallocated MW: {remaining_high:.2f} MW")

display(allocated_df_high["tier"].value_counts().to_frame())

# Conservation of energy assertion
assert abs((total_allocated_high + remaining_high) - surplus_high) < 1e-6, (
    f"Energy balance mismatch: {total_allocated_high} + {remaining_high} != {surplus_high}"
)

Input Surplus: 900.0 MW
Total System-Wide Flexible Capacity: 1142.59 MW
Facilities receiving allocation: 27
Total Allocated MW: 900.00 MW
Remaining Unallocated MW: 0.00 MW


,count
tier,
2,12
1,8
3,7


## 6. Summary comparison across all three tests

Consolidating diagnostic results across the low (100 MW), medium (350 MW), and high (900 MW) surplus test scenarios into a unified summary table.

In [6]:
summary_records = []

for label, s_val, alloc_dict, rem_val in [
    ("Low Surplus", surplus_low, allocations_low, remaining_low),
    ("Medium Surplus", surplus_med, allocations_med, remaining_med),
    ("High Surplus", surplus_high, allocations_high, remaining_high),
]:
    allocated_ids = [fid for fid, mw in alloc_dict.items() if mw > 0.0]
    
    t1_mw = sum(
        mw for fid, mw in alloc_dict.items()
        if df_facilities.loc[df_facilities["facility_id"] == fid, "tier"].values[0] == 1
    )
    t2_mw = sum(
        mw for fid, mw in alloc_dict.items()
        if df_facilities.loc[df_facilities["facility_id"] == fid, "tier"].values[0] == 2
    )
    t3_mw = sum(
        mw for fid, mw in alloc_dict.items()
        if df_facilities.loc[df_facilities["facility_id"] == fid, "tier"].values[0] == 3
    )

    summary_records.append(
        {
            "scenario": label,
            "input_surplus_mw": s_val,
            "allocated_facilities_count": len(allocated_ids),
            "total_allocated_mw": round(sum(alloc_dict.values()), 2),
            "tier1_allocated_mw": round(t1_mw, 2),
            "tier2_allocated_mw": round(t2_mw, 2),
            "tier3_allocated_mw": round(t3_mw, 2),
            "remaining_unallocated_mw": round(rem_val, 2),
        }
    )

display(pd.DataFrame(summary_records))

,scenario,input_surplus_mw,allocated_facilities_count,total_allocated_mw,tier1_allocated_mw,tier2_allocated_mw,tier3_allocated_mw,remaining_unallocated_mw
0,Low Surplus,100.0,5,100.0,100.00,0.00,0.00,0.0
1,Medium Surplus,350.0,20,350.0,196.57,153.43,0.00,0.0
2,High Surplus,900.0,27,900.0,196.57,154.18,549.25,0.0
